# BANTAI — RF-DETR Nano baseline and fine-tuning
**Start with a pothole-trained checkpoint, evaluate it, optionally adapt it, then compare.**

Model: [dronefreak/rdd2022-rfdetr-nano](https://huggingface.co/dronefreak/rdd2022-rfdetr-nano).
Dataset: [publisher's RDD2022 split](https://huggingface.co/datasets/dronefreak/RDD2022),
original [RDD2022 paper](https://doi.org/10.1002/gdj3.260).
Training API: [RF-DETR](https://rfdetr.roboflow.com/learn/train/).

The four model classes are longitudinal crack, transverse crack, alligator crack and
pothole. Potholes are the primary MVP result; we retain all four classes during adaptation.
No application integration is performed here.

## How to run
1. Upload this notebook to Colab. Select a T4 GPU if available.
2. Run sections 1–5 for preparation and the original model's **validation baseline**.
3. In section 6 set `RUN_FINE_TUNING=True` to start the short fine-tuning run.
4. Section 7 compares validation results and chooses a candidate.
5. Once development is finished, enable section 8 for paired final test evaluation.
6. Section 9 exports a results ZIP to send back in our chat.

Data and model caches use `/content` (temporary SSD). Reports and training checkpoints
use Drive. Free GPU access and runtime duration are not guaranteed; training can resume
from a completed epoch, but a partially completed epoch must repeat.

**Evaluation honesty:** preserve the publisher's train/valid/test assignments. Its test
split is a custom held-out portion of publicly labeled RDD images, not the unlabeled
official challenge test. The publisher's split correspondence is documented but not
independently proven here. Near-duplicate/route independence is not guaranteed. Exact
duplicates within the downloaded sample are rejected. Local Philippine camera testing
is still required. Fine-tuning on a small subset of data previously seen by the starting
model is an adaptation experiment; it may not improve performance.

Each numbered section has a troubleshooting slot for a temporary repair cell.
Keep one settings cell. Once a fix works, edit the original section instead of leaving
multiple competing versions. Older notebooks remain separate.


## 1. Configuration and Drive
Defaults download **1200 training + 200 validation + 200 test images**, across
India, Japan, and the Czech Republic. Each split is half pothole-positive and half without
an annotated pothole (the latter can contain cracks). This is an enriched diagnostic sample,
not traffic prevalence. All available road-damage annotations in selected images are retained.

Changing settings requires a new RUN_NAME. The final test remains unused for model
selection. Training is deliberately opt-in in section 6.


In [ ]:
from pathlib import Path
import json, os, sys, subprocess, gc, shutil
RUN_NAME = 'rfdetr_rdd_multicountry_002'
COUNTRY = ('India', 'Japan', 'Czech')
SEED = 42
TRAIN_IMAGES, VALID_IMAGES, TEST_IMAGES = 1200, 200, 200
EPOCHS = 15
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 1e-5
CONFIDENCE = 0.12
assert RUN_NAME.replace('_', '').isalnum(), 'Use letters, numbers and underscores.'
assert all(n >= 10 for n in (TRAIN_IMAGES, VALID_IMAGES, TEST_IMAGES))
assert 0 < CONFIDENCE < 1 and EPOCHS > 0 and BATCH_SIZE > 0 and GRAD_ACCUM_STEPS > 0
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/bantai_rfdetr') / RUN_NAME
OUT = ROOT / 'results'
BACKUP = Path('/content/drive/MyDrive/BANTAI/rfdetr') / RUN_NAME
TRAIN_DIR = BACKUP / 'training'
for folder in (ROOT, OUT, BACKUP, TRAIN_DIR):
    folder.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(ROOT / 'hf_cache')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
print('Temporary work:', ROOT)
print('Persistent results:', BACKUP)


### 1 troubleshooting slot
    If this section fails, insert a code cell **here**, immediately below its main cell.
    Keep the fix limited to this section, then rerun its main cell and dependent sections.
    Record the error and fix in a short Markdown cell. Do not change test selection,
    class mapping or settings to improve test scores. After a fix works, fold it into
    the main cell and clear the temporary cell. Send the section number and error to our chat.


## 2. Dependencies and workflow helpers
This installs exact top-level versions **inside Colab only**. Running this section is
the installation step; no setup scripts from the model repository are executed.
Colab supplies PyTorch; transitive dependencies are resolved by pip and the full
environment is recorded. If imports request a restart, restart the session and rerun
from section 1. Initial installation needs internet even when inference later runs locally.


In [ ]:
PACKAGES = ['rfdetr[train]==1.11.2', 'huggingface-hub==2.3.0', 'pycocotools==2.0.11']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *PACKAGES], check=True)
import torch
from rfdetr import RFDETR, RFDETRNano
from huggingface_hub import hf_hub_download, snapshot_download
from PIL import Image
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU (evaluation only)')
(OUT / 'environment.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')


In [ ]:
"""Standalone helpers embedded in the BANTAI RF-DETR Colab notebook.

Importing this module performs no downloads, package installs, or training.
"""
from pathlib import Path, PurePosixPath
import collections
import hashlib
import json
import math
import random
import shutil
import time

CLASSES = ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']
MODEL_REPO = 'dronefreak/rdd2022-rfdetr-nano'
MODEL_REVISION = '66429f39b916e56c6a33243d776718088f5f43ad'
DATA_REPO = 'dronefreak/RDD2022'
DATA_REVISION = '72b01f94c2e5bfe430861afd73995f2d412b8855'
PACKAGES = ['rfdetr[train]==1.11.2', 'huggingface-hub==2.3.0', 'pycocotools==2.0.11']


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.partial')
    temp.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    temp.replace(path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def identity(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_copy(source, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_name(target.name + '.partial')
    shutil.copyfile(source, temp)
    temp.replace(target)


def backup_results(source, target):
    for path in sorted(Path(source).rglob('*')):
        if path.is_file() and not path.name.endswith('.partial'):
            atomic_copy(path, Path(target) / path.relative_to(source))


def guard_json(path, value):
    path = Path(path)
    if path.exists() and read_json(path) != json.loads(json.dumps(value)):
        raise ValueError('Saved settings/data differ. Choose a new RUN_NAME; do not overwrite this run.')
    write_json(path, value)


def safe_relative(value):
    path = PurePosixPath(value)
    if path.is_absolute() or '..' in path.parts or '\\' in value or ':' in value:
        raise ValueError('Unsafe dataset path')
    return path


def validate_objects(objects, width=None, height=None):
    boxes, labels = objects['bbox'], objects['categories']
    if len(boxes) != len(labels):
        raise ValueError('Box/label counts differ')
    annotations = []
    for box, label in zip(boxes, labels):
        if type(label) is not int or not 0 <= label < len(CLASSES):
            raise ValueError('Unknown class ID')
        if len(box) != 4 or not all(math.isfinite(float(v)) for v in box):
            raise ValueError('Invalid bounding box')
        x, y, w, h = map(float, box)
        if min(x, y) < 0 or min(w, h) <= 0:
            raise ValueError('Invalid box extent')
        if width is not None and (x + w > width or y + h > height):
            raise ValueError('Box extends outside image; inspect source labels')
        annotations.append({'category_id': label, 'bbox': [x, y, w, h],
                            'area': w * h, 'iscrowd': 0})
    return annotations


def select_records(records, count, seed):
    """A fixed half-pothole diagnostic sample, preserving upstream assignments."""
    positive = sorted((r for r in records if 3 in r['objects']['categories']), key=lambda r: r['path'])
    negative = sorted((r for r in records if 3 not in r['objects']['categories']), key=lambda r: r['path'])
    npos, nneg = count // 2, count - count // 2
    if len(positive) < npos or len(negative) < nneg:
        raise ValueError(f'Insufficient images: need {npos} pothole/{nneg} no-pothole; '
                         f'available {len(positive)}/{len(negative)}. Reduce sample sizes in section 1.')
    rng = random.Random(seed)
    return sorted(rng.sample(positive, npos) + rng.sample(negative, nneg), key=lambda r: r['path'])


def metadata_records(cache, country=None):
    records = {split: [] for split in ('train', 'valid', 'test')}
    for path in sorted(Path(cache).glob('data/images/*/shard_*/metadata.jsonl')):
        split = path.parent.parent.name
        if split not in records:
            continue
        for line in path.read_text(encoding='utf-8').splitlines():
            if not line.strip():
                continue
            item = json.loads(line)
            name = item['file_name']
            safe_relative(name)
            if PurePosixPath(name).name != name:
                raise ValueError('Unexpected metadata filename layout')
            if country:
                if isinstance(country, (list, tuple, set)):
                    if not any(name.startswith(c + '_') for c in country):
                        continue
                elif not name.startswith(country + '_'):
                    continue
            validate_objects(item['objects'])
            remote = path.parent.relative_to(cache).as_posix() + '/' + name
            records[split].append({'path': remote, 'file_name': name, 'objects': item['objects']})
    for split, rows in records.items():
        if not rows:
            raise ValueError('No metadata records for ' + split)
        if len({r['path'] for r in rows}) != len(rows):
            raise ValueError('Duplicate metadata entries in ' + split)
    return records


def prepare_dataset(root, backup, counts, seed, country):
    from concurrent.futures import ThreadPoolExecutor
    from huggingface_hub import snapshot_download, hf_hub_download
    from PIL import Image
    root, backup = Path(root), Path(backup)
    cache = root / 'download_cache'
    snapshot_download(DATA_REPO, repo_type='dataset', revision=DATA_REVISION,
                      allow_patterns=['data/images/*/shard_*/metadata.jsonl'], local_dir=cache)
    candidates = metadata_records(cache, country)
    chosen = {s: select_records(candidates[s], n, seed + i)
              for i, (s, n) in enumerate(counts.items())}
    selection = {'dataset': DATA_REPO, 'revision': DATA_REVISION, 'seed': seed,
                 'country': country, 'counts': counts, 'splits': chosen}
    guard_json(backup / 'selection.json', selection)
    prepared = root / 'dataset'
    all_paths = [r['path'] for rows in chosen.values() for r in rows]
    def fetch(path):
        return hf_hub_download(DATA_REPO, filename=path, repo_type='dataset',
                               revision=DATA_REVISION, local_dir=cache)
    with ThreadPoolExecutor(max_workers=6) as pool:
        for i, _ in enumerate(pool.map(fetch, all_paths)):
            if (i + 1) % 50 == 0:
                print(i + 1, '/', len(all_paths), 'images ready', flush=True)
    hashes = {}
    exclusions = []
    manifest = {'dataset': DATA_REPO, 'revision': DATA_REVISION, 'splits': {}, 'classes': CLASSES}
    for split, rows in chosen.items():
        folder = prepared / split
        folder.mkdir(parents=True, exist_ok=True)
        coco = {'info': {'description': 'BANTAI RDD2022 diagnostic subset'}, 'licenses': [],
                'images': [], 'annotations': [],
                'categories': [{'id': i, 'name': n, 'supercategory': 'road_damage'}
                               for i, n in enumerate(CLASSES)]}
        manifest['splits'][split] = []
        coverage = collections.Counter()
        for i, row in enumerate(rows):
            src = cache.joinpath(*safe_relative(row['path']).parts)
            digest = sha256(src)
            if digest in hashes:
                raise ValueError('Exact duplicate image in selected sample: ' + row['path']
                                 + ' and ' + hashes[digest] + '. Revise selection before proceeding.')
            with Image.open(src) as image:
                image.load()
                orientation = image.getexif().get(274, 1)
                if orientation != 1:
                    exclusions.append({'split': split, 'path': row['path'], 'sha256': digest,
                                       'orientation': orientation,
                                       'reason': 'EXIF rotation/mirroring requires annotation review; excluded without transforming pixels or boxes.'})
                    print('Quarantined:', row['path'], '| EXIF orientation:', orientation, flush=True)
                    write_json(root / 'results/data_exclusions.json', exclusions)
                    write_json(backup / 'data_exclusions.json', exclusions)
                    continue
                width, height = image.size
            hashes[digest] = row['path']
            anns = validate_objects(row['objects'], width, height)
            filename = f'{i:05d}.jpg'
            atomic_copy(src, folder / filename)
            coco['images'].append({'id': i, 'file_name': filename, 'width': width, 'height': height})
            for ann in anns:
                coverage[ann['category_id']] += 1
                coco['annotations'].append({**ann, 'id': len(coco['annotations']) + 1, 'image_id': i})
            manifest['splits'][split].append({**row, 'id': i, 'local_file': filename,
                                              'width': width, 'height': height, 'sha256': digest})
        if not coco['images'] or not coverage[3]:
            raise ValueError('No usable pothole evaluation/training data in ' + split)
        if split == 'train' and set(coverage) != set(range(4)):
            raise ValueError('Training sample must include all four classes; increase TRAIN_IMAGES.')
        write_json(folder / '_annotations.coco.json', coco)
        print(split, len(coco['images']), 'usable images of', len(rows), 'requested; boxes:',
              {CLASSES[c]: n for c, n in sorted(coverage.items())})
    write_json(root / 'results/data_exclusions.json', exclusions)
    write_json(backup / 'data_exclusions.json', exclusions)
    manifest['id'] = identity(manifest)
    guard_json(backup / 'manifest.json', manifest)
    write_json(root / 'results/manifest.json', manifest)
    return prepared, manifest


def box_iou(a, b):
    overlap = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - overlap
    return overlap / union if union > 0 else 0.0


def match_boxes(truth, predictions, threshold=.5):
    unmatched = set(range(len(truth)))
    tp = 0
    for prediction in sorted(predictions, key=lambda p: p['score'], reverse=True):
        choices = [(box_iou(prediction['xyxy'], truth[i]), i) for i in unmatched]
        overlap, index = max(choices, default=(0., -1))
        if index >= 0 and overlap >= threshold:
            unmatched.remove(index)
            tp += 1
    return {'tp': tp, 'fp': len(predictions)-tp, 'fn': len(truth)-tp}


def fixed_metrics(coco, rows, confidence):
    truth = collections.defaultdict(list)
    for ann in coco['annotations']:
        x, y, w, h = ann['bbox']
        truth[(ann['image_id'], ann['category_id'])].append([x, y, x+w, y+h])
    output = {}
    for category, name in enumerate(CLASSES):
        totals = collections.Counter()
        bg, alarms = 0, 0
        for row in rows:
            labels = truth[(row['image_id'], category)]
            predictions = [p for p in row['predictions'] if p['category_id'] == category and p['score'] >= confidence]
            totals.update(match_boxes(labels, predictions))
            if not labels:
                bg += 1
                alarms += bool(predictions)
        tp, fp, fn = (totals[k] for k in ('tp', 'fp', 'fn'))
        precision = tp / (tp+fp) if tp+fp else 0.
        recall = tp / (tp+fn) if tp+fn else 0.
        output[name] = {**dict(totals), 'precision': precision, 'recall': recall,
                        'f1': 2*precision*recall/(precision+recall) if precision+recall else 0.,
                        'images_without_class': bg, 'false_alarm_images': alarms,
                        'image_false_alarm_rate': alarms/bg if bg else None}
    return output


def validate_model_class_id(value, image, detected_ids):
    """Reject unknown classes without silently dropping or remapping predictions."""
    try:
        class_id = int(value)
        valid = (
            not isinstance(value, bool)
            and class_id == value
            and 0 <= class_id < len(CLASSES)
        )
    except (TypeError, ValueError, OverflowError):
        valid = False
    if valid:
        return class_id
    expected = dict(enumerate(CLASSES))
    raise ValueError(
        f"Unexpected model class ID: {value!r}; "
        f"image_id={image.get('id')!r}; file={image.get('file_name')!r}; "
        f"detected IDs in this image={detected_ids!r}; "
        f"expected mapping={expected!r}. "
        "Evaluation stopped to avoid incorrect metrics. Inspect the checkpoint's "
        "class-name mapping before changing IDs. Do not subtract 1 or discard "
        "detections without confirming that mapping."
    )


def foreground_predictions(detections, image):
    """Exclude only RF-DETR's explicitly named final background slot."""
    boxes = detections.xyxy.tolist()
    ids = detections.class_id.tolist()
    scores = detections.confidence.tolist()
    names = detections.data.get('class_name')
    if not (len(boxes) == len(ids) == len(scores)):
        raise ValueError('Detection array lengths do not match')
    if names is not None and len(names) != len(ids):
        raise ValueError('Detection class-name array length does not match')
    predictions = []
    background_filtered = 0
    for i, (box, cls, score) in enumerate(zip(boxes, ids, scores)):
        name = names[i] if names is not None else None
        # Confirmed from the supplied installed predict() implementation:
        # the final slot is returned with the explicit __background__ label.
        if cls == len(CLASSES) and name == '__background__':
            background_filtered += 1
            continue
        class_id = validate_model_class_id(cls, image, ids)
        predictions.append({'xyxy': box, 'category_id': class_id, 'score': float(score)})
    return predictions, background_filtered


def evaluate(detector, weights, prepared, split, out, backup, manifest_id, confidence, label):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    from PIL import Image
    import torch
    out, backup, prepared = Path(out) / label, Path(backup) / label, Path(prepared)
    out.mkdir(parents=True, exist_ok=True)
    annotation_file = prepared / split / '_annotations.coco.json'
    coco = read_json(annotation_file)
    signature = identity({'model': sha256(weights), 'manifest': manifest_id, 'split': split,
                          'annotations': sha256(annotation_file), 'confidence': confidence,
                          'floor': .001, 'version': 'rfdetr-eval-v1'})
    guard_json(backup / 'identity.json', {'id': signature})
    rows = []
    # Warm up once; reported timings exclude this, IO, download and metric calculation.
    first = prepared / split / coco['images'][0]['file_name']
    detector.predict(str(first), threshold=.001)
    for i, image in enumerate(coco['images']):
        target = out / 'predictions' / f"{image['id']:05d}.json"
        saved = backup / 'predictions' / target.name
        if not target.exists() and saved.exists():
            atomic_copy(saved, target)
        if target.exists():
            row = read_json(target)
            if row['signature'] != signature or row['image_id'] != image['id']:
                raise ValueError('Prediction cache mismatch; use a new RUN_NAME')
        else:
            with Image.open(prepared / split / image['file_name']) as source:
                rgb = source.convert('RGB')
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            start = time.perf_counter()
            detections = detector.predict(rgb, threshold=.001)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - start
            predictions, background_filtered = foreground_predictions(detections, image)
            row = {'signature': signature, 'image_id': image['id'],
                   'predictions': predictions, 'predict_seconds': elapsed,
                   'background_filtered': background_filtered}
            write_json(target, row)
            atomic_copy(target, saved)
        rows.append(row)
        if (i+1) % 25 == 0:
            print(label, i+1, '/', len(coco['images']), 'saved', flush=True)
    ground_truth = COCO(str(annotation_file))
    detections = []
    for row in rows:
        for p in row['predictions']:
            x1, y1, x2, y2 = p['xyxy']
            detections.append({'image_id': row['image_id'], 'category_id': p['category_id'],
                               'score': p['score'], 'bbox': [x1, y1, x2-x1, y2-y1]})
    if detections:
        predicted = ground_truth.loadRes(detections)
    else:
        predicted = COCO()
        predicted.dataset = {**coco, 'annotations': []}
        predicted.createIndex()
    evaluator = COCOeval(ground_truth, predicted, 'bbox')
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()
    per_class = {}
    values = evaluator.eval['precision']
    for c, name in enumerate(CLASSES):
        all_iou = values[:, :, c, 0, -1]
        iou50 = values[0, :, c, 0, -1]
        per_class[name] = {'map50_95': float(all_iou[all_iou > -1].mean()) if (all_iou > -1).any() else None,
                           'map50': float(iou50[iou50 > -1].mean()) if (iou50 > -1).any() else None}
    report = {'signature': signature, 'label': label, 'split': split, 'images': len(rows),
              'background_filtered': sum(r.get('background_filtered', 0) for r in rows),
              'model_sha256': sha256(weights), 'map50_95': float(evaluator.stats[0]),
              'map50': float(evaluator.stats[1]), 'per_class_ap': per_class,
              'operating_confidence': confidence, 'matching_iou': .5,
              'fixed_confidence': fixed_metrics(coco, rows, confidence),
              'mean_predict_ms': sum(r['predict_seconds'] for r in rows)/len(rows)*1000,
              'speed_note': 'Warm-start predict call, excludes file IO; cached records may span sessions.',
              'map_note': 'COCO maxDets=100, low prediction floor=0.001.'}
    write_json(out / 'summary.json', report)
    write_json(out / 'coco_predictions.json', detections)
    save_examples(prepared, split, coco, rows, out / 'examples', confidence)
    backup_results(out, backup)
    return report


def save_examples(prepared, split, coco, rows, destination, confidence):
    from PIL import Image, ImageDraw, ImageOps
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    counts = collections.Counter()
    index = []
    for image, row in zip(coco['images'], rows):
        anns = [a for a in coco['annotations'] if a['image_id'] == image['id'] and a['category_id'] == 3]
        truth = [[a['bbox'][0], a['bbox'][1], a['bbox'][0]+a['bbox'][2], a['bbox'][1]+a['bbox'][3]] for a in anns]
        predictions = [p for p in row['predictions'] if p['category_id'] == 3 and p['score'] >= confidence]
        match = match_boxes(truth, predictions)
        kind = 'miss' if match['fn'] else 'false_alarm' if match['fp'] else 'correct' if match['tp'] else 'background'
        if counts[kind] >= 3:
            continue
        counts[kind] += 1
        with Image.open(Path(prepared) / split / image['file_name']) as source:
            picture = source.convert('RGB')
        draw = ImageDraw.Draw(picture)
        for box in truth:
            draw.rectangle(box, outline='lime', width=3)
        for p in predictions:
            draw.rectangle(p['xyxy'], outline='red', width=2)
            draw.text(tuple(p['xyxy'][:2]), f"{p['score']:.2f}", fill='red')
        name = f"{kind}_{image['id']:05d}.jpg"
        ImageOps.contain(picture, (1280, 960)).save(destination / name)
        index.append({'file': name, 'image_id': image['id'], **match})
    write_json(destination / 'index.json', index)


In [ ]:
config = dict(version='rfdetr-workflow-v1', packages=PACKAGES, country=COUNTRY, seed=SEED,
              counts={'train': TRAIN_IMAGES, 'valid': VALID_IMAGES, 'test': TEST_IMAGES},
              epochs=EPOCHS, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
              lr=LEARNING_RATE, lr_encoder=1e-6, confidence=CONFIDENCE,
              model_repo=MODEL_REPO, model_revision=MODEL_REVISION,
              data_repo=DATA_REPO, data_revision=DATA_REVISION)
guard_json(BACKUP / 'config.json', config)
write_json(OUT / 'config.json', config)
backup_results(OUT, BACKUP / 'reports')


### 2 troubleshooting slot
    If this section fails, insert a code cell **here**, immediately below its main cell.
    Keep the fix limited to this section, then rerun its main cell and dependent sections.
    Record the error and fix in a short Markdown cell. Do not change test selection,
    class mapping or settings to improve test scores. After a fix works, fold it into
    the main cell and clear the temporary cell. Send the section number and error to our chat.


## 3. Download and verify the starting checkpoint
Pin both model and dataset revisions so future updates do not silently change this run.
Weights are approximately 121 MB. Load through RF-DETR's checkpoint-aware loader,
which restores saved architecture settings. Restricted checkpoint loading stays enabled;
if it refuses an unfamiliar checkpoint, stop and inspect rather than enabling unsafe loading.


In [ ]:
WEIGHTS = Path(hf_hub_download(MODEL_REPO, filename='checkpoint_best_total.pth',
                             revision=MODEL_REVISION, local_dir=ROOT / 'model'))
model_config_path = hf_hub_download(MODEL_REPO, filename='config.json',
                                   revision=MODEL_REVISION, local_dir=ROOT / 'model')
publisher_config = read_json(model_config_path)
assert publisher_config['class_names'] == CLASSES, 'Publisher class mapping changed.'
model_info = dict(repo=MODEL_REPO, revision=MODEL_REVISION, sha256=sha256(WEIGHTS),
                  class_names=CLASSES, resolution=publisher_config['model_config']['resolution'])
assert model_info['resolution'] == 384, 'Review unexpected Nano resolution before continuing.'
guard_json(BACKUP / 'model.json', model_info)
write_json(OUT / 'model.json', model_info)

def load_detector(path):
    detector = RFDETR.from_checkpoint(str(path), device=DEVICE)
    assert isinstance(detector, RFDETRNano), 'Checkpoint is not RF-DETR Nano.'
    assert detector.model_config.num_classes == 4, 'Unexpected class count.'
    names = detector.model.class_names
    assert names == CLASSES, f'Unexpected checkpoint class names: {names}'
    return detector

detector = load_detector(WEIGHTS)
print('Loaded RF-DETR Nano:', model_info)
backup_results(OUT, BACKUP / 'reports')


### 3 troubleshooting slot
    If this section fails, insert a code cell **here**, immediately below its main cell.
    Keep the fix limited to this section, then rerun its main cell and dependent sections.
    Record the error and fix in a short Markdown cell. Do not change test selection,
    class mapping or settings to improve test scores. After a fix works, fold it into
    the main cell and clear the temporary cell. Send the section number and error to our chat.


## 4. Download only the chosen images and prepare COCO labels
Metadata is small; only selected images are downloaded, using six concurrent requests.
All image work uses Colab's SSD. Upstream split membership is preserved; test data is
prepared but not used for training or parameter selection. Image dimensions, boxes,
class IDs and exact duplicates are checked. Source metadata and hashes are exported.
Images with EXIF rotation/mirroring are quarantined with an exclusion record rather
than altering pixels or guessing how to transform annotation boxes. Usable counts
may therefore be lower than requested; review the counts and exclusions.
Related scenes crossing the publisher's splits remain a limitation requiring further review.


In [ ]:
PREPARED, manifest = prepare_dataset(ROOT, BACKUP, config['counts'], SEED, COUNTRY)
backup_results(OUT, BACKUP / 'reports')
# Smoke test on one TRAIN image; no test image is displayed.
detector = load_detector(WEIGHTS)
first = read_json(PREPARED / 'train/_annotations.coco.json')['images'][0]
smoke = detector.predict(str(PREPARED / 'train' / first['file_name']), threshold=CONFIDENCE)
print('Inference smoke test completed; predicted boxes:', len(smoke))
print('Dataset ready:', PREPARED)


### 4 troubleshooting slot
    If this section fails, insert a code cell **here**, immediately below its main cell.
    Keep the fix limited to this section, then rerun its main cell and dependent sections.
    Record the error and fix in a short Markdown cell. Do not change test selection,
    class mapping or settings to improve test scores. After a fix works, fold it into
    the main cell and clear the temporary cell. Send the section number and error to our chat.


## 5. Original model: validation baseline
This checks the starting model without training. mAP uses a low prediction floor;
precision, recall, F1 and false alarms use the fixed operating confidence in section 1.
Per-image results save immediately to Drive and resume after interruptions.
Green example boxes are pothole labels; red boxes are pothole predictions.


In [ ]:
baseline_valid = evaluate(detector, WEIGHTS, PREPARED, 'valid', OUT, BACKUP / 'reports',
                          manifest['id'], CONFIDENCE, 'baseline_validation')
print(json.dumps({'pothole_AP': baseline_valid['per_class_ap']['pothole'],
                  'pothole_fixed': baseline_valid['fixed_confidence']['pothole'],
                  'mean_predict_ms': baseline_valid['mean_predict_ms']}, indent=2))


### 5 troubleshooting slot
    If this section fails, insert a code cell **here**, immediately below its main cell.
    Keep the fix limited to this section, then rerun its main cell and dependent sections.
    Record the error and fix in a short Markdown cell. Do not change test selection,
    class mapping or settings to improve test scores. After a fix works, fold it into
    the main cell and clear the temporary cell. Send the section number and error to our chat.


## 6. Optional fine-tuning — run after reviewing the baseline
Set RUN_FINE_TUNING=True below to start. GPU is required by this notebook for training.
The default is 15 epochs, early stopping after 4 stagnant validation epochs, batch 2,
gradient accumulation 8, multi-scale zoom training, and EMA weight smoothing.
Improvements are not guaranteed.

Images remain on SSD. RF-DETR writes each epoch's recovery checkpoint directly to
Drive (`training/last.ckpt`), with best weights promoted at successful completion.
Drive writes add overhead but avoid a custom background backup process. Recovery
replays an incomplete epoch. Keep sufficient Drive space for optimizer checkpoints.

Only train and valid data are provided to the training call: a separate training view
contains those two folders, with no test folder. Test evaluation is disabled in training.
Once final testing starts, this run cannot be trained again. Use a new study run for
further development and independently reserve new test data before interpreting it as a fresh test.


In [ ]:
RUN_FINE_TUNING = False  # Change to True when ready to spend GPU time.
fine_tuned_path = TRAIN_DIR / 'checkpoint_best_total.pth'
if RUN_FINE_TUNING:
    assert DEVICE == 'cuda', 'GPU unavailable. Baseline evaluation works on CPU; defer training.'
    assert not (BACKUP / 'final_test_started.json').exists() or (TRAIN_DIR / 'completed.json').exists(), 'Final test already opened; training is locked.'
    assert (OUT / 'baseline_validation/summary.json').exists(), 'Run section 5 first.'
    training_identity = dict(config=config, manifest=manifest['id'], model=model_info['sha256'])
    guard_json(TRAIN_DIR / 'run_identity.json', training_identity)
    if (TRAIN_DIR / 'completed.json').exists():
        assert fine_tuned_path.exists(), 'Completed run is missing its best weights.'
        print('Fine-tuning already completed; reusing saved weights.')
    else:
        # Give the trainer train/valid only; it must never see our test images.
        training_view = ROOT / 'training_view'
        training_view.mkdir(exist_ok=True)
        for split in ('train', 'valid'):
            link = training_view / split
            if not link.exists():
                link.symlink_to(PREPARED / split, target_is_directory=True)
        assert not (training_view / 'test').exists()
        if 'detector' in globals():
            del detector
        gc.collect()
        torch.cuda.empty_cache()
        trainer_model = load_detector(WEIGHTS)
        resume = TRAIN_DIR / 'last.ckpt'
        print('Resuming from completed epoch' if resume.exists() else 'Starting from downloaded road-damage weights')
        trainer_model.train(dataset_dir=str(training_view), output_dir=str(TRAIN_DIR),
                            epochs=EPOCHS, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
                            lr=LEARNING_RATE, lr_encoder=1e-6, resolution=384, device=DEVICE,
                            num_workers=0, seed=SEED, class_names=CLASSES,
                            early_stopping=True, early_stopping_patience=4,
                            early_stopping_min_delta=0.001, early_stopping_use_ema=True,
                            use_ema=True, checkpoint_interval=5,
                            run_test=False, tensorboard=False, wandb=False,
                            multi_scale=True, expanded_scales=False,
                            resume=str(resume) if resume.exists() else None)
        assert fine_tuned_path.exists(), 'Best checkpoint missing; inspect the training output before continuing.'
        write_json(TRAIN_DIR / 'completed.json', {'sha256': sha256(fine_tuned_path)})
        del trainer_model
        gc.collect()
        torch.cuda.empty_cache()
else:
    print('Fine-tuning skipped. Set RUN_FINE_TUNING=True in this cell to enable it.')


### 6 troubleshooting slot
    If this section fails, insert a code cell **here**, immediately below its main cell.
    Keep the fix limited to this section, then rerun its main cell and dependent sections.
    Record the error and fix in a short Markdown cell. Do not change test selection,
    class mapping or settings to improve test scores. After a fix works, fold it into
    the main cell and clear the temporary cell. Send the section number and error to our chat.


## 7. Validation comparison and candidate selection
Compare both models on validation data. The candidate is selected by pothole
mAP@50:95, breaking ties in favor of the original model. This does not claim that
overall crack performance improved; those scores are reported separately.


In [ ]:
completed = TRAIN_DIR / 'completed.json'
candidate = 'baseline'
comparison = {'baseline': baseline_valid}
if completed.exists():
    assert read_json(completed)['sha256'] == sha256(fine_tuned_path), 'Fine-tuned weights changed.'
    if 'detector' in globals():
        del detector
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    detector = load_detector(fine_tuned_path)
    tuned_valid = evaluate(detector, fine_tuned_path, PREPARED, 'valid', OUT, BACKUP / 'reports',
                           manifest['id'], CONFIDENCE, 'finetuned_validation')
    comparison['fine_tuned'] = tuned_valid
    original_ap = baseline_valid['per_class_ap']['pothole']['map50_95']
    tuned_ap = tuned_valid['per_class_ap']['pothole']['map50_95']
    assert original_ap is not None and tuned_ap is not None, 'No evaluable pothole labels.'
    if tuned_ap > original_ap:
        candidate = 'fine_tuned'
comparison['selected_by_validation'] = candidate
write_json(OUT / 'validation_comparison.json', comparison)
backup_results(OUT, BACKUP / 'reports')
for name in ('baseline', 'fine_tuned'):
    if name in comparison:
        print(name, 'pothole mAP50:95:', comparison[name]['per_class_ap']['pothole']['map50_95'])
print('Validation-selected candidate:', candidate)


### 7 troubleshooting slot
    If this section fails, insert a code cell **here**, immediately below its main cell.
    Keep the fix limited to this section, then rerun its main cell and dependent sections.
    Record the error and fix in a short Markdown cell. Do not change test selection,
    class mapping or settings to improve test scores. After a fix works, fold it into
    the main cell and clear the temporary cell. Send the section number and error to our chat.


## 8. Final paired test — enable only when development is finished
Set RUN_FINAL_TEST=True to evaluate both original and fine-tuned weights on the
same reserved test sample. This freezes the candidate selected in section 7. Do not
change training settings or select confidence thresholds using test results.
Rerunning after interruption resumes saved predictions. Scores measure this sample,
not the official challenge leaderboard or local barangay deployment accuracy.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    frozen = {'manifest': manifest['id'], 'config': identity(config),
              'baseline_sha256': model_info['sha256'], 'candidate': candidate,
              'fine_tuned_sha256': sha256(fine_tuned_path) if completed.exists() else None}
    guard_json(BACKUP / 'final_test_started.json', frozen)
    final = {'selected_before_test': candidate, 'results': {}}
    pairs = [('baseline', WEIGHTS)]
    if completed.exists():
        pairs.append(('fine_tuned', fine_tuned_path))
    for name, weights in pairs:
        if 'detector' in globals():
            del detector
        gc.collect()
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        detector = load_detector(weights)
        final['results'][name] = evaluate(detector, weights, PREPARED, 'test', OUT, BACKUP / 'reports',
                                          manifest['id'], CONFIDENCE, name + '_test')
    write_json(OUT / 'final_comparison.json', final)
    backup_results(OUT, BACKUP / 'reports')
    for name, report in final['results'].items():
        print(name, 'POTHOLE:', report['per_class_ap']['pothole'], report['fixed_confidence']['pothole'])
else:
    print('Test remains reserved. Enable RUN_FINAL_TEST only after finishing development.')


### 8 troubleshooting slot
    If this section fails, insert a code cell **here**, immediately below its main cell.
    Keep the fix limited to this section, then rerun its main cell and dependent sections.
    Record the error and fix in a short Markdown cell. Do not change test selection,
    class mapping or settings to improve test scores. After a fix works, fold it into
    the main cell and clear the temporary cell. Send the section number and error to our chat.


## 8A. Upload a road photo — demonstration
Run after section 7, before section 9. Section 8 may remain disabled. This cell
explicitly reloads the candidate selected in section 7 and verifies its weight hash.
It does not assume the detector left by any earlier cell is the chosen model.
Upload a downloaded JPG, PNG or WebP road image using the file picker. Boxes show all
four road-damage classes with confidence scores. This is an exploratory demonstration;
it does not modify test metrics or train the model. Outputs are included in section 9's ZIP.


In [ ]:
#@title 8A. Upload road photos - validation-selected model
# Paste this entire cell after section 7 and before section 9 in the running Colab notebook.
# Section 8 may remain disabled; this demo never evaluates the reserved dataset.
# It requires the corrected section 2 helpers and the existing notebook configuration.
from pathlib import Path
import gc
import hashlib
import io
from google.colab import files
from IPython.display import display
from PIL import Image, ImageDraw, ImageOps

# Read the recorded validation selection, without requiring final-test results.
validation_path = OUT / 'validation_comparison.json'
if not validation_path.exists() and (BACKUP / 'reports/validation_comparison.json').exists():
    atomic_copy(BACKUP / 'reports/validation_comparison.json', validation_path)
assert validation_path.exists(), 'Section 7 validation comparison is missing.'
validation_report = read_json(validation_path)
selected_candidate = validation_report['selected_by_validation']
assert selected_candidate in ('baseline', 'fine_tuned'), 'Unknown validation-selected candidate.'
print('Section 8A | Validation-selected candidate:', selected_candidate)

# Explicitly reload the validation-selected checkpoint. Do not reuse section 8's detector.
if selected_candidate == 'fine_tuned':
    demo_weights = TRAIN_DIR / 'checkpoint_best_total.pth'
    expected_sha256 = validation_report['fine_tuned']['model_sha256']
else:
    demo_weights = Path(WEIGHTS)
    expected_sha256 = validation_report['baseline']['model_sha256']
assert demo_weights.exists(), 'Selected checkpoint is missing. Restore it before running the demo.'
assert sha256(demo_weights) == expected_sha256, 'Selected checkpoint hash differs from validation.'
for model_variable in ('detector', 'demo_detector'):
    if model_variable in globals():
        del globals()[model_variable]
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
demo_detector = load_detector(demo_weights)
print('Demo model:', selected_candidate, '| confidence:', CONFIDENCE)
print('Exploratory demonstration only; these images do not change evaluation scores or select the model.')

uploaded = files.upload()
assert uploaded, 'No image uploaded. Rerun this cell to choose a JPG, PNG, or WebP image.'
demo_dir = OUT / 'upload_demo'
demo_dir.mkdir(parents=True, exist_ok=True)
colors = ['#00bfff', '#ffb000', '#d65dff', '#ff4242']
for filename, image_bytes in uploaded.items():
    try:
        with Image.open(io.BytesIO(image_bytes)) as source_image:
            road_image = ImageOps.exif_transpose(source_image).convert('RGB')
    except (OSError, ValueError):
        print('Skipped unreadable image:', filename)
        continue
    detections = demo_detector.predict(road_image, threshold=CONFIDENCE)
    predictions, background_filtered = foreground_predictions(detections, {'file_name': filename, 'id': 'upload-demo'})
    annotated = road_image.copy()
    draw = ImageDraw.Draw(annotated)
    line_width = max(2, round(min(annotated.size) / 300))
    for prediction in predictions:
        category = prediction['category_id']
        x1, y1, x2, y2 = prediction['xyxy']
        label = f"{CLASSES[category].replace('_', ' ')} {prediction['score']:.1%}"
        color = colors[category]
        draw.rectangle((x1, y1, x2, y2), outline=color, width=line_width)
        text_position = (max(0, x1), max(0, y1 - 14))
        text_box = draw.textbbox(text_position, label)
        draw.rectangle(text_box, fill=color)
        draw.text(text_position, label, fill='black')
    image_hash = hashlib.sha256(image_bytes).hexdigest()
    image_path = demo_dir / (image_hash[:16] + '_detections.jpg')
    annotated.save(image_path)
    write_json(image_path.with_suffix('.json'), {
        'source_filename': filename, 'source_sha256': image_hash,
        'selected_by_validation': selected_candidate, 'checkpoint_sha256': expected_sha256,
        'confidence': CONFIDENCE, 'predictions': predictions,
        'background_filtered': background_filtered, 'purpose': 'exploratory_upload_demo'})
    print(filename, '|', len(predictions), 'road-damage detections')
    if not predictions:
        print('No road damage detected at this confidence; this does not prove the road is defect-free.')
        low_detections = demo_detector.predict(road_image, threshold=0.01)
        low_preds, _ = foreground_predictions(low_detections, {'file_name': filename, 'id': 'upload-demo-low'})
        if low_preds:
            top = max(low_preds, key=lambda p: p['score'])
            print(f"  Top low-confidence candidate: {CLASSES[top['category_id']]} @ {top['score']:.1%}")
    for prediction in predictions:
        print(' ', CLASSES[prediction['category_id']], f"{prediction['score']:.1%}")
    display(ImageOps.contain(annotated, (1400, 1000)))
backup_results(demo_dir, BACKUP / 'reports/upload_demo')
print('Demo saved. Run section 9 next to export the results ZIP, including these examples.')


### 8A troubleshooting slot
    If this section fails, insert a code cell **here**, immediately below its main cell.
    Keep the fix limited to this section, then rerun its main cell and dependent sections.
    Record the error and fix in a short Markdown cell. Do not change test selection,
    class mapping or settings to improve test scores. After a fix works, fold it into
    the main cell and clear the temporary cell. Send the section number and error to our chat.


### 8A diagnostic: investigate a missed detection (optional)
Run only to investigate uploaded demonstration images. Reuses the original uploaded
photo and explicitly compares both evaluated checkpoints. Prints the strongest score
for each class and shows up to five boxes at display thresholds 0.25, 0.10 and 0.05.
Low-score boxes are uncertain hypotheses. This cell does not change CONFIDENCE,
select a candidate, retrain weights, or alter final-test reports. Outputs enter the ZIP.


In [ ]:
# Paste as one code cell under Section 8A, before Section 9. No retraining.
from pathlib import Path
import gc
import hashlib
import io
from google.colab import files
from IPython.display import display
from PIL import Image, ImageDraw, ImageOps

# Reuse the original uploaded image bytes when available, not the screenshot/overlay.
diagnostic_uploads = globals().get('uploaded')
if not diagnostic_uploads:
    diagnostic_uploads = files.upload()
assert diagnostic_uploads, 'Upload the same original road photo used in the missed detection.'
validation_file = OUT / 'validation_comparison.json'
if not validation_file.exists() and (BACKUP / 'reports/validation_comparison.json').exists():
    atomic_copy(BACKUP / 'reports/validation_comparison.json', validation_file)
assert validation_file.exists(), 'Restore the Section 7 validation comparison first.'
diagnostic_validation = read_json(validation_file)
diagnostic_pairs = [('baseline', Path(WEIGHTS)),
                    ('fine_tuned', TRAIN_DIR / 'checkpoint_best_total.pth')]
for model_name, path in diagnostic_pairs:
    assert path.exists(), f'Missing {model_name} checkpoint: restore it before comparing.'
    assert model_name in diagnostic_validation, f'Missing {model_name} validation report.'
    assert sha256(path) == diagnostic_validation[model_name]['model_sha256'], 'Checkpoint differs from evaluated weights.'

print('DIAGNOSTIC ONLY: evaluation confidence and final-test scores remain unchanged.')
print('Low-score boxes are uncertain hypotheses, not confirmed road defects.')
diagnostic_dir = OUT / 'diagnostic_demo'
diagnostic_dir.mkdir(parents=True, exist_ok=True)
diagnostic_records = {}
diagnostic_colors = ['#00bfff', '#ffb000', '#d65dff', '#ff4242']

for model_name, path in diagnostic_pairs:
    for variable in ('detector', 'demo_detector', 'diagnostic_detector'):
        if variable in globals():
            del globals()[variable]
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    diagnostic_detector = load_detector(path)
    for filename, raw_image in diagnostic_uploads.items():
        try:
            with Image.open(io.BytesIO(raw_image)) as source:
                original_image = ImageOps.exif_transpose(source).convert('RGB')
        except (OSError, ValueError):
            print('Skipped unreadable image:', filename)
            continue
        detection = diagnostic_detector.predict(original_image, threshold=0.001)
        predictions, filtered = foreground_predictions(detection, {'id': 'diagnostic', 'file_name': filename})
        predictions = sorted(predictions, key=lambda item: item['score'], reverse=True)
        image_hash = hashlib.sha256(raw_image).hexdigest()
        diagnostic_records.setdefault(image_hash, {'filename': filename, 'models': {}})
        diagnostic_records[image_hash]['models'][model_name] = {
            'checkpoint_sha256': sha256(path), 'predictions_above_0_001': predictions,
            'background_filtered': filtered}
        print('\n', model_name.upper(), '|', filename)
        for category, class_name in enumerate(CLASSES):
            ranked = [p for p in predictions if p['category_id'] == category]
            maximum = f"{ranked[0]['score']:.2%}" if ranked else 'none at or above 0.001'
            print(class_name, '| strongest score:', maximum,
                  '| boxes at 0.25:', sum(p['score'] >= .25 for p in ranked))

        # Same photo at three display thresholds; max five boxes per panel avoids clutter.
        sheet = Image.new('RGB', (1500, 470), 'white')
        for column, display_threshold in enumerate((0.25, 0.10, 0.05)):
            annotated = original_image.copy()
            draw = ImageDraw.Draw(annotated)
            shown = [p for p in predictions if p['score'] >= display_threshold][:5]
            for p in shown:
                label = f"{CLASSES[p['category_id']]} {p['score']:.1%}"
                color = diagnostic_colors[p['category_id']]
                draw.rectangle(p['xyxy'], outline=color, width=max(2, min(annotated.size)//250))
                position = (max(0, p['xyxy'][0]), max(0, p['xyxy'][1]-14))
                draw.rectangle(draw.textbbox(position, label), fill=color)
                draw.text(position, label, fill='black')
            thumbnail = ImageOps.contain(annotated, (490, 420))
            sheet.paste(thumbnail, (column*500, 40))
            ImageDraw.Draw(sheet).text((column*500+5, 10),
                f'{model_name}: score >= {display_threshold:.2f} (top 5)', fill='black')
        sheet.save(diagnostic_dir / f'{image_hash[:16]}_{model_name}.jpg')
        display(sheet)

write_json(diagnostic_dir / 'comparison.json', {
    'purpose': 'exploratory missed-detection investigation; not evaluation or threshold selection',
    'inference_floor': .001, 'display_thresholds': [.25, .10, .05],
    'images': diagnostic_records})
backup_results(diagnostic_dir, BACKUP / 'reports/diagnostic_demo')
print('Saved diagnostic comparison. Run Section 9 and send the ZIP plus these panels here.')


## 9. Export results for review
The ZIP includes completed reports, per-image predictions, manifests, environment
versions and representative pothole mistakes. It excludes the raw dataset, model
weights and large optimizer checkpoints. Training checkpoints remain in your Drive.
You can export after baseline evaluation, after training, or after final testing; the
status file explicitly records which phases finished. Send the ZIP back in our chat.

If Colab disconnects: open the **same notebook**, keep RUN_NAME unchanged, and rerun
sections 1–5. Data is rebuilt, completed predictions are restored, and section 6 resumes
from `last.ckpt` when enabled. Do not create duplicate sections for recovery.


In [ ]:
status = {'baseline_validation_completed': (OUT / 'baseline_validation/summary.json').exists(),
          'fine_tuning_completed': (TRAIN_DIR / 'completed.json').exists(),
          'final_test_started': (BACKUP / 'final_test_started.json').exists(),
          'final_test_completed': (OUT / 'final_comparison.json').exists(),
          'limitations': ['Publisher split provenance not independently proven.',
                          'Enriched sample; no natural-prevalence claim.',
                          'Near-duplicate/route independence not guaranteed.',
                          'Local camera performance remains untested.']}
write_json(OUT / 'status.json', status)
# Include lightweight training records, never optimizer state or large weights.
for filename in ('training_config.json', 'completed.json', 'run_identity.json'):
    if (TRAIN_DIR / filename).exists():
        atomic_copy(TRAIN_DIR / filename, OUT / 'training' / filename)
backup_results(OUT, BACKUP / 'reports')
archive = Path(shutil.make_archive(str(ROOT / (RUN_NAME + '_results')), 'zip', OUT))
atomic_copy(archive, BACKUP / archive.name)
print('Saved to Drive:', BACKUP / archive.name)
print(json.dumps(status, indent=2))
from google.colab import files
files.download(str(archive))


### 9 troubleshooting slot
    If this section fails, insert a code cell **here**, immediately below its main cell.
    Keep the fix limited to this section, then rerun its main cell and dependent sections.
    Record the error and fix in a short Markdown cell. Do not change test selection,
    class mapping or settings to improve test scores. After a fix works, fold it into
    the main cell and clear the temporary cell. Send the section number and error to our chat.
